# Setup and Installation

In [ ]:
! pip install langchain -q
! pip install langchain-community -q
! pip install langchain-openai -q
! pip install langchain-google-genai -q
! pip install langchain-experimental -q
! pip install neo4j -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 7.7 MB/s eta 0:00:00


# Initialize OpenAI LLm

In [ ]:
from google.colab import userdata
import os

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# set OpenAI API key
os.environ ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

# Initialize the GenAI model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

# Initialize Embedding Model

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Load Data

In [ ]:
from langchain_core.documents import Document

text = Document(page_content = """Three students, A, B, and C, are tackling two subjects, X (Mathematics) and Y (Literature). Each has a unique perspective, weaving their experiences into a shared academic journey. A, gifted in Mathematics, thrives on solving equations but struggles with the abstract world of poetry and storytelling. On the other hand, B shines in Literature, captivating others with a flair for creative writing, yet finds numbers daunting and formulas perplexing.
C, a generalist, performs decently in both subjects but often bridges gaps between A and B. While A helps B understand mathematical concepts, B guides A through essay writing. Meanwhile, C organizes group study sessions, offering real-world examples to connect ideas from X and Y, making both subjects more relatable. Their collaboration not only enhances their learning but fosters a sense of camaraderie, demonstrating the power of teamwork in overcoming challenges.
""")

# Split Documents into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
chunks = splitter.split_documents([text])

In [ ]:
chunks

[Document(metadata={}, page_content='Three students, A, B, and C, are tackling two subjects, X (Mathematics) and Y (Literature). Each has a unique perspective, weaving their experiences into a shared academic journey. A, gifted in Mathematics, thrives on solving equations but struggles'),
 Document(metadata={}, page_content='equations but struggles with the abstract world of poetry and storytelling. On the other hand, B shines in Literature, captivating others with a flair for creative writing, yet finds numbers daunting and formulas perplexing.'),
 Document(metadata={}, page_content='C, a generalist, performs decently in both subjects but often bridges gaps between A and B. While A helps B understand mathematical concepts, B guides A through essay writing. Meanwhile, C organizes group study sessions, offering real-world examples'),
 Document(metadata={}, page_content='offering real-world examples to connect ideas from X and Y, making both subjects more relatable. Their collaboration n

# Graph Initialization and Transformation

In [ ]:
from langchain_community.graphs import Neo4jGraph
import os

neo4j_uri = 'neo4j+s://abbddae2.databases.neo4j.io'
neo4j_username = 'abbddae2'
neo4j_password = 'TWpdEynjkyPJKBNFJEW8aUvKbtbIButO3NJgvAo2ips'
os.environ['NEO4J_URI'] = neo4j_uri
os.environ['NEO4J_USERNAME'] = neo4j_username
os.environ['NEO4J_PASSWORD'] = neo4j_password

# initialize Neo4J graph database
graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    database=neo4j_username
)

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
graph_transformer = LLMGraphTransformer(llm=llm)

In [ ]:
# convert text chunks to graph nodes
graph_documents = graph_transformer.convert_to_graph_documents(chunks)

In [ ]:
len(graph_docuemnts)

4

In [ ]:
graph_docuemnts[0]

GraphDocument(nodes=[Node(id='A', type='Person', properties={}), Node(id='B', type='Person', properties={}), Node(id='C', type='Person', properties={}), Node(id='Mathematics', type='Subject', properties={}), Node(id='Literature', type='Subject', properties={}), Node(id='Equations', type='Concept', properties={})], relationships=[Relationship(source=Node(id='A', type='Person', properties={}), target=Node(id='Mathematics', type='Subject', properties={}), type='TACKLES', properties={}), Relationship(source=Node(id='B', type='Person', properties={}), target=Node(id='Mathematics', type='Subject', properties={}), type='TACKLES', properties={}), Relationship(source=Node(id='C', type='Person', properties={}), target=Node(id='Mathematics', type='Subject', properties={}), type='TACKLES', properties={}), Relationship(source=Node(id='A', type='Person', properties={}), target=Node(id='Literature', type='Subject', properties={}), type='TACKLES', properties={}), Relationship(source=Node(id='B', type=

In [ ]:
for node in graph_docuemnts[0].nodes:
  print(node)

id='A' type='Person' properties={}
id='B' type='Person' properties={}
id='C' type='Person' properties={}
id='Mathematics' type='Subject' properties={}
id='Literature' type='Subject' properties={}
id='Equations' type='Concept' properties={}


In [ ]:
for relationship in graph_docuemnts[0].relationships:
  print(relationship)

source=Node(id='A', type='Person', properties={}) target=Node(id='Mathematics', type='Subject', properties={}) type='TACKLES' properties={}
source=Node(id='B', type='Person', properties={}) target=Node(id='Mathematics', type='Subject', properties={}) type='TACKLES' properties={}
source=Node(id='C', type='Person', properties={}) target=Node(id='Mathematics', type='Subject', properties={}) type='TACKLES' properties={}
source=Node(id='A', type='Person', properties={}) target=Node(id='Literature', type='Subject', properties={}) type='TACKLES' properties={}
source=Node(id='B', type='Person', properties={}) target=Node(id='Literature', type='Subject', properties={}) type='TACKLES' properties={}
source=Node(id='C', type='Person', properties={}) target=Node(id='Literature', type='Subject', properties={}) type='TACKLES' properties={}
source=Node(id='A', type='Person', properties={}) target=Node(id='Mathematics', type='Subject', properties={}) type='GIFTED_IN' properties={}
source=Node(id='A', t

In [ ]:
# add nodes and relationships to graph
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel = True,
    include_source = True
    )

In [ ]:
# indexing enables fast searches within text-based properties
def create_fulltext_ndex(g):
  cypher = "CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:_Entity_) ON EACH [e.id]"
  g.query(cypher)


In [ ]:
create_fulltext_ndex(graph)

# Querying the Graph and Entity Retrieval

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

entity_prompt = ChatPromptTemplate.from_messages(
    [(
        "system","You are extracting entities from the input question. If the question refers to 'X' as a subject, extract 'Mathematics'. If it refers to 'Y' as a subject, extract 'Literature'. Otherwise, extract entities as they appear.",
    ),
    (
        "human","Use the following information to extract entities"
        "input:{question}",
    ),
    ]
)

In [ ]:
from pydantic import BaseModel,Field
from typing import List

class Entities(BaseModel):
  names:List[str] = Field (
      ...,
      description = "All the entities that appear in the text"
  )

In [ ]:
entity_chain = (
    entity_prompt
    | llm.with_structured_output(Entities)
)

In [ ]:
entity_chain = (
    entity_prompt
    | llm.with_structured_output(Entities)
)

# Graph Retriever

In [ ]:
def remove_lucene_chars(text: str) -> str:
    # Lucene special characters
    special_chars = (
        '+', '-', '&', '|', '!', '(', ')', '{', '}', '[', ']', '^', '~', '*', '?', ':', '\\', '/'
    )
    # Escape all special characters
    for char in special_chars:
        text = text.replace(char, f'\\{char}')
    return text

def generate_full_text_query(input):
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

def graph_retriever(question: str) -> str:
    entities = entity_chain.invoke({"question": question})
    extracted_entities = entities.names

    if not extracted_entities:
        return "No relevant entities found in the question."

    # Handle multiple entities (e.g., "who learn both X and Y")
    if len(extracted_entities) > 1:
        person_var = "p"
        exists_conditions = []
        params = {}

        for i, entity_name in enumerate(extracted_entities):
            entity_node_alias = f"e{i}"
            param_name = f'entity_id_{i}'
            exists_conditions.append(
                f"EXISTS {{ MATCH ({person_var})-[]-({entity_node_alias}) WHERE {entity_node_alias}.id = ${param_name} }}"
            )
            params[param_name] = entity_name

        combined_exists_conditions = " AND ".join(exists_conditions)
        final_cypher_query = f"""MATCH ({person_var}:Person)
WHERE {combined_exists_conditions}
RETURN DISTINCT {person_var}.id AS personId"""

        response = graph.query(final_cypher_query, params)
        if response:
            return "\n".join([el['personId'] for el in response])
        else:
            return "No persons found learning all specified subjects."
    else: # Handle a single entity query
        entity = extracted_entities[0]
        # Original query logic for single entity
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 20
            """,
            {"query": generate_full_text_query(entity)},
        )
        return "\n".join([el['output'] for el in response])

In [ ]:
print(graph_retriever("who learn both X and Y subjects"))

No persons found learning all specified subjects.


# Semantic Search Retriever

In [ ]:
!pip install --upgrade --force-reinstall langchain-neo4j langchain-experimental -q
from langchain_neo4j import Neo4jVector

vector_index = Neo4jVector.from_existing_graph(
    embedding_model,
    search_type = "hybrid",
    node_label = "Document",
    text_node_properties = ["text"],
    embedding_node_property = "embedding",
)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.7.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.


ClientError: {neo4j_code: Neo.ClientError.Database.DatabaseNotFound} {message: Unable to get a routing table for database 'neo4j' because this database does not exist} {gql_status: 22000} {gql_status_description: error: data exception}

In [ ]:
vector_index.similarity_search("who learn both X and Y subjects", k= 2)

In [ ]:
def retriever(question):
  graph_search_result = graph_retriever(question)
  semantic_search_result = [data.page_content for data in vector_index.similarity_search(question, k=2)]
  final_data = f"Graph data:{graph_search_result}\nText data:{' '.join(sementic_search_result)}"
  return final_data

# Define Prompt Template for RAG

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system"."Answer this question using the provided context only.",
        ),
        (
            "human","Context":{context}"
            "Question:"{question}","
        ),
    ]
)

# Create RAG Chain

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {
        "context" : retriever,
        "question": RunnablePassthrough(),
    }
    | propmt
    | llm
)


# Invoke RAG Chain wth Example Questions

In [ ]:
response = chain.invoke("who learn both X and Y subjects")
print(response.content)

In [ ]:
response = chain.invoke("who are the students in that school")
print(response.content)